# 第17章 量化策略与回测 —— 配套代码

对应正文 `book/part4/17-backtesting.md`。依赖内置数据，**离线可跑**。

**本 notebook 演示内容**：
1. 环境初始化与数据加载
2. 20日动量信号生成
3. 向量化回测核心：`position = signal.shift(1)`
4. 策略净值曲线与基准对比
5. 加入A股双边交易成本
6. 完整绩效指标表
7. 净值 + 水下曲线双子图
8. **前视偏差演示**：有前视 vs 无前视净值对比
9. 换手率分析与成本敏感性
10. 参数扫描（动量窗口稳健性）
11. 样本内/样本外对比
12. 习题参考解答

> **提示**：先运行第1个代码格（Cell 1）生成数据，其余格均依赖该格输出。

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

In [ ]:
# Cell 1：环境初始化与数据加载
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from fds import (
    load_sample_prices, load_market, daily_returns, set_chinese_font,
    annualized_return, annualized_volatility, sharpe_ratio, max_drawdown
)

np.random.seed(42)
set_chinese_font()

prices  = load_sample_prices()
market  = load_market()
returns = daily_returns(prices)

TRADING_DAYS = 252
RF = 0.02

print(f'价格数据维度：{prices.shape}')
print(f'日期范围：{prices.index[0].date()} ~ {prices.index[-1].date()}')
print(f'资产列表：{list(prices.columns)}')
prices.tail()

## 15.2 信号生成：20日动量

动量策略（Momentum）以20个交易日为回望窗口：

$$\text{Signal}_t = \begin{cases} 1 & \text{若过去20日收益} > 0 \\\\ 0 & \text{其他}\end{cases}$$

信号在 $t$ 日收盘后生成，$t+1$ 日才能成交（T+1规则）。

In [ ]:
# Cell 2：动量信号生成
WINDOW = 20

momentum = prices.pct_change(WINDOW)
signal = (momentum > 0).astype(float)

print('各资产做多信号占比：')
print((signal == 1).mean().round(3))
print('\n信号示例（末10行）：')
print(signal.tail(10).to_string())

## 15.3 向量化回测核心：`position = signal.shift(1)`

**最重要的一行代码**——对应T+1制度，消除前视偏差：

$$\text{策略日收益}_t = \text{Position}_{t-1} \times r_t$$

In [ ]:
# Cell 3：向量化回测（不含成本）

# 核心一行：今日持仓 = 昨日信号（T+1规则）
position = signal.shift(1)

gross_ret = (position * returns).dropna()
nav_gross = (1 + gross_ret).cumprod()
nav_bh    = (1 + returns.loc[gross_ret.index]).cumprod()

print('策略毛收益描述统计：')
print(gross_ret.describe().round(4))
print('\n策略净值末日值：')
print(nav_gross.iloc[-1].round(3).to_string())

## 15.4 加入A股双边交易成本

A股双边成本（简化）：佣金 0.025%+印花税 0.05%（仅卖）+滑点 0.025% = **单边约 0.1%**

$$\text{年化成本} = \text{年化换手率} \times C_{单边}$$

In [ ]:
# Cell 4：加入交易成本
COST = 0.001  # 单边成本率 0.1%

# position.diff() 会使索引多一个NaN行，用reindex对齐到gross_ret的日期
turnover   = position.diff().abs()
daily_cost = turnover.reindex(gross_ret.index).fillna(0) * COST
net_ret    = gross_ret - daily_cost
nav_net    = (1 + net_ret).cumprod()

# 换手率统计（用有效换手数据）
to_valid = position.diff().abs().dropna()
print('各资产日均换手率：')
print(to_valid.mean().round(4).to_string())
print('\n各资产年化换手率：')
print((to_valid.mean() * 252).round(1).to_string())
print('\n各资产年化成本（%）：')
print((to_valid.mean() * 252 * COST * 100).round(2).to_string())

## 15.5 完整绩效指标表

评估维度：**年化收益 / 年化波动 / 夏普 / 最大回撤 / 卡尔玛 / 胜率 / 换手率 / 成本占比**

In [ ]:
# Cell 5：完整绩效指标表

def perf_metrics(rets, pos=None, rf=0.02, cost=0.001):
    ann_r  = annualized_return(rets)
    ann_v  = annualized_volatility(rets)
    sp     = sharpe_ratio(rets, risk_free=rf)
    mdd    = max_drawdown(rets)
    calmar = ann_r / abs(mdd) if mdd != 0 else float('nan')
    win    = float((rets > 0).mean())
    d = {'年化收益': ann_r, '年化波动': ann_v, '夏普比率': sp,
         '最大回撤': mdd, '卡尔玛比率': calmar, '胜率': win}
    if pos is not None:
        to_d = pos.diff().abs().reindex(rets.index).fillna(0)
        to_a = float(to_d.mean()) * 252
        ac   = to_a * cost
        d['年化换手率'] = to_a
        d['年化成本']   = ac
        d['成本占收益比'] = ac / ann_r if ann_r != 0 else float('nan')
    return d

rows = []
for ticker in returns.columns:
    m = perf_metrics(net_ret[ticker], pos=position[ticker], rf=RF, cost=COST)
    m['资产'] = ticker
    rows.append(m)

df_perf = pd.DataFrame(rows).set_index('资产')
print('=' * 60)
print('20日动量策略绩效（含0.1%单边成本）')
print('=' * 60)
print(df_perf.round(3).to_string())

## 15.6 净值曲线 + 水下曲线

左列：策略净值 vs 买入持有；右列：水下曲线（回撤深度）。

In [ ]:
# Cell 6：净值 + 水下曲线（四资产）
fig, axes = plt.subplots(4, 2, figsize=(15, 14))
fig.suptitle('20日动量策略：净值曲线 + 水下曲线（含0.1%单边成本）', fontsize=13, fontweight='bold')

for i, ticker in enumerate(returns.columns):
    ax_nav = axes[i, 0]
    ax_dd  = axes[i, 1]
    nav_s  = nav_net[ticker]
    nav_b  = nav_bh[ticker].reindex(nav_s.index)
    dd_s   = nav_s / nav_s.cummax() - 1
    mdd_val = max_drawdown(net_ret[ticker])
    sr_val  = sharpe_ratio(net_ret[ticker], risk_free=RF)

    nav_s.plot(ax=ax_nav, color='steelblue', lw=1.5, label='策略（含成本）')
    nav_b.plot(ax=ax_nav, color='gray', lw=1, linestyle='--', alpha=0.7, label='买入持有')
    ax_nav.set_title(f'{ticker}  |  夏普={sr_val:.2f}  MDD={mdd_val:.1%}')
    ax_nav.set_ylabel('净值')
    ax_nav.legend(fontsize=8)
    ax_nav.axhline(1, color='black', lw=0.5, linestyle=':')

    dd_s.plot(ax=ax_dd, color='crimson', lw=1)
    ax_dd.fill_between(dd_s.index, dd_s.values, 0, color='crimson', alpha=0.2)
    ax_dd.axhline(mdd_val, color='black', lw=1, linestyle='--', label=f'MDD={mdd_val:.1%}')
    ax_dd.set_title(f'{ticker} 水下曲线')
    ax_dd.set_ylabel('回撤')
    ax_dd.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax_dd.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 15.7 前视偏差演示：有前视 vs 无前视净值对比

**回测陷阱警告**：少了一行 `shift(1)`，净值会**虚假地高出很多**，
这是初学者最常犯、也最难察觉的错误。

In [ ]:
# Cell 7：前视偏差演示

# 有前视偏差（错误：忘记shift）
signal_biased = (momentum > 0).astype(float)
gross_biased  = (signal_biased * returns).dropna()

gross_correct = gross_ret.copy()
common_idx    = gross_biased.index.intersection(gross_correct.index)
nav_biased    = (1 + gross_biased.loc[common_idx]).cumprod()
nav_correct_a = (1 + gross_correct.loc[common_idx]).cumprod()

print('=== 前视偏差对比 ===')
for ticker in returns.columns:
    r_bias = annualized_return(gross_biased[ticker].loc[common_idx])
    r_corr = annualized_return(gross_correct[ticker].loc[common_idx])
    print(f'{ticker}: 有前视={r_bias:.2%}，无前视={r_corr:.2%}，虚假超额={r_bias - r_corr:.2%}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, ticker in enumerate(['TECH', 'LIQUOR']):
    ax = axes[i]
    nav_biased[ticker].plot(ax=ax, color='crimson', lw=1.5, label='有前视偏差（错误）')
    nav_correct_a[ticker].plot(ax=ax, color='steelblue', lw=1.5, label='无前视偏差（正确）')
    ax.fill_between(
        common_idx,
        nav_biased[ticker].values,
        nav_correct_a[ticker].values,
        alpha=0.2, color='crimson', label='虚假超额部分'
    )
    ax.set_title(f'{ticker}：前视偏差的影响')
    ax.set_ylabel('净值')
    ax.legend(fontsize=9)
    ax.axhline(1, color='black', lw=0.5, linestyle=':')

plt.suptitle('前视偏差演示：红色区域为虚假收益（由错误实现产生）', fontsize=12, color='crimson')
plt.tight_layout()
plt.show()
print('\n结论：前视偏差让净值虚高，是回测中最常见且最危险的错误之一。')

## 15.8 成本敏感性分析

扫描单边成本率 0%~0.3%，找出使夏普=0的临界成本。

In [ ]:
# Cell 8：成本敏感性分析

cost_grid = np.arange(0, 0.0031, 0.0002)

cost_results = {}
for ticker in returns.columns:
    sharpes_list = []
    for c in cost_grid:
        to = position[ticker].diff().abs()
        dc = to.reindex(gross_ret.index).fillna(0) * c
        nr = gross_ret[ticker] - dc
        sharpes_list.append(sharpe_ratio(nr, risk_free=RF))
    cost_results[ticker] = sharpes_list

print('各资产使夏普降为0的临界单边成本率：')
for ticker in returns.columns:
    s = pd.Series(cost_results[ticker], index=cost_grid)
    zc = None
    for j in range(len(s)-1):
        if s.iloc[j] >= 0 and s.iloc[j+1] < 0:
            x0, x1 = s.index[j], s.index[j+1]
            y0, y1 = s.iloc[j], s.iloc[j+1]
            zc = x0 - y0 * (x1 - x0) / (y1 - y0)
            break
    if zc is not None:
        print(f'  {ticker}: 临界成本={zc*100:.3f}%')
    else:
        print(f'  {ticker}: 扫描范围内未降为0')

fig, ax = plt.subplots(figsize=(10, 5))
for ticker in returns.columns:
    ax.plot(cost_grid * 100, cost_results[ticker], marker='o', ms=3, label=ticker)
ax.axhline(0, color='black', lw=1, linestyle='--', alpha=0.5)
ax.axvline(0.1, color='gray', lw=1, linestyle=':', alpha=0.7, label='0.1%（本文默认）')
ax.set_xlabel('单边成本率 (%)')
ax.set_ylabel('年化夏普比率')
ax.set_title('成本敏感性：不同成本率下的夏普比率')
ax.legend()
plt.tight_layout()
plt.show()

## 15.9 参数扫描：动量窗口的稳健性

扫描窗口 W in {5, 10, 15, 20, 25, 30, 40, 60}，
判断结果是**参数高原**（真实信号）还是**参数孤岛**（过拟合）。

In [ ]:
# Cell 9：参数扫描

windows = [5, 10, 15, 20, 25, 30, 40, 60]

scan_results = {}
for w in windows:
    mom_w = prices.pct_change(w)
    sig_w = (mom_w > 0).astype(float)
    pos_w = sig_w.shift(1)
    gr_w  = (pos_w * returns).dropna()
    to_w  = pos_w.diff().abs().reindex(gr_w.index).fillna(0)
    nr_w  = gr_w - to_w * COST
    scan_results[w] = sharpe_ratio(nr_w, risk_free=RF).to_dict()

df_scan = pd.DataFrame(scan_results).T
df_scan.index.name = '动量窗口（日）'
print('参数扫描结果（夏普比率）：')
print(df_scan.round(3).to_string())

fig, ax = plt.subplots(figsize=(11, 5))
for ticker in returns.columns:
    ax.plot(windows, df_scan[ticker].values, marker='o', ms=5, lw=1.5, label=ticker)
ax.axhline(0, color='black', lw=1, linestyle='--', alpha=0.5)
ax.axvline(20, color='gray', lw=1, linestyle=':', label='W=20（本章选用）')
ax.set_xlabel('动量窗口（交易日）')
ax.set_ylabel('年化夏普比率')
ax.set_title('参数扫描：动量窗口 vs 夏普比率')
ax.legend()
ax.set_xticks(windows)
plt.tight_layout()
plt.show()
print('\n若曲线平稳 -> 参数高原（稳健）；若只有孤立峰 -> 过拟合风险。')

## 15.10 样本内/样本外对比

- **样本内（IS）**：前70%数据，用于参数探索
- **样本外（OOS）**：后30%数据，独立验证

样本外结果看到后不得再调整参数。

In [ ]:
# Cell 10：样本内/样本外分析

T = len(prices)
split = int(T * 0.7)
split_date = prices.index[split]

print(f'样本内：{prices.index[0].date()} ~ {prices.index[split-1].date()} （{split}日）')
print(f'样本外：{split_date.date()} ~ {prices.index[-1].date()} （{T-split}日）')

rows_io = []
for ticker in returns.columns:
    for seg, label in [('IS', '样本内'), ('OOS', '样本外')]:
        nr = net_ret[ticker].loc[:split_date] if seg == 'IS' else net_ret[ticker].loc[split_date:]
        if len(nr) < 20:
            continue
        rows_io.append({
            '资产': ticker, '阶段': label,
            '年化收益': annualized_return(nr),
            '夏普比率': sharpe_ratio(nr, risk_free=RF),
            '最大回撤': max_drawdown(nr),
        })

df_io = pd.DataFrame(rows_io)
print('\n样本内 vs 样本外绩效：')
print(df_io.pivot_table(index='资产', columns='阶段',
      values=['年化收益', '夏普比率', '最大回撤']).round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
x = np.arange(len(returns.columns))
w = 0.35
is_sp  = df_io[df_io['阶段']=='样本内'].set_index('资产')['夏普比率']
oos_sp = df_io[df_io['阶段']=='样本外'].set_index('资产')['夏普比率']
ax.bar(x - w/2, is_sp.values,  w, label='样本内', color='steelblue', alpha=0.8)
ax.bar(x + w/2, oos_sp.values, w, label='样本外', color='darkorange', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(returns.columns)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('年化夏普比率')
ax.set_title('样本内 vs 样本外：夏普比率对比')
ax.legend()

ax2 = axes[1]
for ticker in returns.columns:
    nav_net[ticker].plot(ax=ax2, lw=1.2, label=ticker)
ax2.axvline(split_date, color='black', lw=1.5, linestyle='--')
ax2.set_title('全段净值（左=样本内，右=样本外）')
ax2.set_ylabel('净值')
ax2.legend(fontsize=8)
ax2.axhline(1, color='black', lw=0.5, linestyle=':')
plt.tight_layout()
plt.show()

## 15.11 综合可视化：四资产完整回测总览

In [ ]:
# Cell 11：四资产完整回测总览
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('20日动量策略：四资产完整回测总览', fontsize=13, fontweight='bold')

for ax, ticker in zip(axes.flat, returns.columns):
    nav_g = nav_gross[ticker]
    nav_n = nav_net[ticker]
    nav_b = nav_bh[ticker].reindex(nav_n.index)

    nav_g.plot(ax=ax, color='steelblue', lw=1.2, linestyle='--', alpha=0.7, label='毛收益（无成本）')
    nav_n.plot(ax=ax, color='navy',      lw=1.8, label='策略（含成本）')
    nav_b.plot(ax=ax, color='gray',      lw=1,   linestyle=':', alpha=0.7, label='买入持有')

    sr    = sharpe_ratio(net_ret[ticker], risk_free=RF)
    mdd_v = max_drawdown(net_ret[ticker])
    ann_r = annualized_return(net_ret[ticker])

    ax.set_title(f'{ticker}  |  Sharpe={sr:.2f}  MDD={mdd_v:.1%}  年化={ann_r:.1%}')
    ax.set_ylabel('净值')
    ax.legend(fontsize=8)
    ax.axhline(1, color='black', lw=0.5, linestyle=':')

plt.tight_layout()
plt.show()

## 用 akquant 框架做事件驱动回测（可选）

前面是手写的**向量化**回测；下面用 [akquant](https://github.com/akfamily/akquant)（akshare 生态、Rust 内核）做**事件驱动**回测——逐 bar 撮合，更贴近真实交易。

> 需先安装：`uv sync --extra quant`（或 `pip install akquant`）。未安装时本格会自动跳过。实盘把 `df` 换成 akshare 真实行情即可。

In [ ]:
# akquant 事件驱动回测（可选：需 uv sync --extra quant）
try:
    import akquant as aq
    from akquant import Strategy

    # akquant 需要 OHLC；内置数据只有收盘价，据此构造示意开高低
    close = prices["LIQUOR"]
    df = close.rename("close").reset_index()          # 列：date, close
    df["open"] = close.shift(1).bfill().to_numpy()
    df["high"], df["low"], df["volume"] = close * 1.01, close * 0.99, 1000

    class MaCross(Strategy):
        def on_bar(self, bar):
            pos = self.get_position(bar.symbol)
            if pos == 0 and bar.close > bar.open:        # 阳线买入
                self.buy(symbol=bar.symbol, quantity=100)
            elif pos > 0 and bar.close < bar.open:        # 阴线平仓
                self.close_position(symbol=bar.symbol)

    result = aq.run_backtest(data=df, strategy=MaCross, initial_cash=100000.0, symbols="LIQUOR")
    print(result)
    result.equity_curve.plot(figsize=(9, 4), title="akquant 事件驱动回测净值曲线")
    plt.ylabel("账户净值"); plt.xlabel("Bar"); plt.tight_layout(); plt.show()
except ImportError:
    print("未安装 akquant，跳过本格。安装：uv sync --extra quant（或 pip install akquant）")

## 15.12 习题参考解答

对应正文第 17.13 节习题，可直接运行验证。

In [ ]:
# 习题15.1：5日/20日均线策略 vs 20日动量策略
print('=' * 55)
print('习题15.1：均线金叉/死叉 vs 动量策略（TECH）')
print('=' * 55)

ticker_ex = 'TECH'
ma5_  = prices[ticker_ex].rolling(5).mean()
ma20_ = prices[ticker_ex].rolling(20).mean()
sig_ma = (ma5_ > ma20_).astype(float)
pos_ma = sig_ma.shift(1)
gr_ma  = (pos_ma * returns[ticker_ex]).dropna()
to_ma  = pos_ma.diff().abs().reindex(gr_ma.index).fillna(0)
nr_ma  = gr_ma - to_ma * COST

nr_mom = net_ret[ticker_ex]
idx_c  = nr_ma.index.intersection(nr_mom.index)

result = pd.DataFrame({
    '均线策略': [f"{annualized_return(nr_ma.loc[idx_c]):.2%}",
               f"{sharpe_ratio(nr_ma.loc[idx_c], RF):.3f}",
               f"{max_drawdown(nr_ma.loc[idx_c]):.2%}"],
    '动量策略': [f"{annualized_return(nr_mom.loc[idx_c]):.2%}",
               f"{sharpe_ratio(nr_mom.loc[idx_c], RF):.3f}",
               f"{max_drawdown(nr_mom.loc[idx_c]):.2%}"],
}, index=['年化收益（含成本）', '夏普比率', '最大回撤'])
print(result.to_string())

In [ ]:
# 习题15.2：前视偏差量化
print('=' * 55)
print('习题15.2：前视偏差量化影响')
print('=' * 55)

print('各资产虚假超额（有前视 - 无前视 年化收益）：')
for ticker in returns.columns:
    r_bias = annualized_return(gross_biased[ticker].loc[common_idx])
    r_corr = annualized_return(gross_correct[ticker].loc[common_idx])
    print(f'  {ticker}: 虚假超额={r_bias - r_corr:.2%}')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
t = 'TECH'
nav_biased[t].plot(ax=ax1, color='crimson', lw=1.5, label='有前视（错误）')
nav_correct_a[t].plot(ax=ax1, color='steelblue', lw=1.5, label='无前视（正确）')
ax1.set_ylabel('净值')
ax1.legend()
ax1.set_title('TECH：前视偏差导致的净值虚高（习题15.2）')

nav_diff = nav_biased[t] - nav_correct_a[t]
nav_diff.plot(ax=ax2, color='darkorange', lw=1.5)
ax2.fill_between(nav_diff.index, nav_diff.values, 0, color='darkorange', alpha=0.3)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_ylabel('净值之差（前视溢价）')
ax2.set_xlabel('日期')
ax2.set_title('前视溢价随时间的累积')
plt.tight_layout()
plt.show()

In [ ]:
# 习题15.3：成本-夏普关系，找临界成本
print('=' * 55)
print('习题15.3：成本-夏普比率关系分析（TECH）')
print('=' * 55)

ticker_ex3 = 'TECH'
cost_fine  = np.arange(0, 0.0041, 0.0001)

sharpes_ex3 = []
for c in cost_fine:
    to3 = position[ticker_ex3].diff().abs().reindex(gross_ret.index).fillna(0)
    nr3 = gross_ret[ticker_ex3] - to3 * c
    sharpes_ex3.append(sharpe_ratio(nr3, risk_free=RF))

s_arr  = np.array(sharpes_ex3)
zero_c = None
for j in range(len(s_arr)-1):
    if s_arr[j] >= 0 and s_arr[j+1] < 0:
        x0, x1 = cost_fine[j], cost_fine[j+1]
        zero_c = x0 - s_arr[j] * (x1 - x0) / (s_arr[j+1] - s_arr[j])
        break

if zero_c is not None:
    print(f'临界单边成本率：{zero_c*100:.3f}%')
else:
    print('扫描范围内夏普未降为0')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cost_fine * 100, sharpes_ex3, color='steelblue', lw=2)
ax.axhline(0, color='black', lw=1, linestyle='--')
ax.axvline(0.1, color='gray', lw=1, linestyle=':', label='0.1%（默认）')
if zero_c is not None:
    ax.axvline(zero_c * 100, color='crimson', lw=1.5, linestyle='--',
               label=f'临界={zero_c*100:.3f}%')
ax.set_xlabel('单边成本率 (%)')
ax.set_ylabel('夏普比率')
ax.set_title(f'{ticker_ex3}：成本-夏普关系（习题15.3）')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 习题15.4：参数扫描 + 样本内/样本外
print('=' * 55)
print('习题15.4：参数扫描 + 样本内/样本外对比')
print('=' * 55)

windows_ex4 = [5, 10, 15, 20, 25, 30, 40, 60]
split_ex4   = int(len(prices) * 0.6)
split_date4 = prices.index[split_ex4]

scan_is4, scan_oos4 = [], []
for w in windows_ex4:
    mom_w = prices['TECH'].pct_change(w)
    sig_w = (mom_w > 0).astype(float)
    pos_w = sig_w.shift(1)
    gr_w  = (pos_w * returns['TECH']).dropna()
    to_w  = pos_w.diff().abs().reindex(gr_w.index).fillna(0)
    nr_w  = gr_w - to_w * COST
    nr_is4  = nr_w.loc[:split_date4]
    nr_oos4 = nr_w.loc[split_date4:]
    scan_is4.append(float(sharpe_ratio(nr_is4,  RF)) if len(nr_is4)>5  else float('nan'))
    scan_oos4.append(float(sharpe_ratio(nr_oos4, RF)) if len(nr_oos4)>5 else float('nan'))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(windows_ex4, scan_is4,  marker='o', ms=5, lw=1.5, color='steelblue',  label='样本内（前60%）')
ax.plot(windows_ex4, scan_oos4, marker='s', ms=5, lw=1.5, color='darkorange', label='样本外（后40%）', linestyle='--')
ax.axhline(0, color='black', lw=0.8, linestyle=':')
ax.axvline(20, color='gray', lw=1, linestyle=':', alpha=0.7, label='W=20')
ax.set_xlabel('动量窗口（交易日）')
ax.set_ylabel('年化夏普比率')
ax.set_title('TECH：参数扫描样本内 vs 样本外（习题15.4）')
ax.legend()
ax.set_xticks(windows_ex4)
plt.tight_layout()
plt.show()

print(f'样本内最优窗口：{windows_ex4[int(np.nanargmax(scan_is4))]}')
print(f'样本外最优窗口：{windows_ex4[int(np.nanargmax(scan_oos4))]}')
print('若两者接近：策略稳健；若差异大：过拟合风险。')

In [ ]:
# 习题15.5：随机策略与数据窥探
print('=' * 55)
print('习题15.5：随机策略 + 数据窥探演示')
print('=' * 55)

rng_ex5 = np.random.default_rng(42)
N_TRIALS = 500

random_sharpes = []
for _ in range(N_TRIALS):
    rand_sig = pd.DataFrame(
        rng_ex5.integers(0, 2, size=prices.shape),
        index=prices.index, columns=prices.columns
    ).astype(float)
    rand_pos = rand_sig.shift(1)
    rand_gr  = (rand_pos * returns).dropna()
    rand_to  = rand_pos.diff().abs().reindex(rand_gr.index).fillna(0)
    rand_nr  = rand_gr - rand_to * COST
    best_sr  = float(sharpe_ratio(rand_nr, risk_free=RF).max())
    random_sharpes.append(best_sr)

random_sharpes = np.array(random_sharpes)
true_best_sr   = float(sharpe_ratio(net_ret, risk_free=RF).max())
pctile         = float((random_sharpes < true_best_sr).mean())

print(f'随机策略（{N_TRIALS}次）：均值={random_sharpes.mean():.3f}，最大={random_sharpes.max():.3f}')
print(f'真实动量策略（最佳资产）夏普：{true_best_sr:.3f}')
print(f'排名分位：{pctile:.1%}（越高说明越不像纯运气）')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(random_sharpes, bins=30, color='steelblue', alpha=0.6,
        edgecolor='white', label=f'{N_TRIALS}个随机策略')
ax.axvline(true_best_sr, color='crimson', lw=2, label=f'真实动量={true_best_sr:.2f}')
ax.set_xlabel('夏普比率')
ax.set_ylabel('频次')
ax.set_title('数据窥探演示：随机策略夏普分布（习题15.5）')
ax.legend()
plt.tight_layout()
plt.show()

print('\n结论：多重检验在随机数据上也能产生高夏普。')
print('Bonferroni校正：测试n策略时，单次p值阈值调整为0.05/n。')